# 📡 TelecomX LATAM — Análisis de Evasión de Clientes (Churn)

**Challenge 2 — Data Science LATAM | Alura**

---

## 🎯 Objetivo del Proyecto

Este proyecto implementa un proceso completo de **ETL (Extract, Transform, Load)** sobre datos de clientes de una empresa de telecomunicaciones ficticia, **TelecomX**. El objetivo final es identificar los factores que explican la **evasión de clientes (Churn)** y generar recomendaciones estratégicas para reducirla.

## 🗂️ Etapas del Proyecto

| Etapa | Descripción |
|-------|-------------|
| **1. Extracción** | Obtención de datos desde API en formato JSON |
| **2. Transformación** | Limpieza, normalización y enriquecimiento del dataset |
| **3. Carga** | Exportación del dataset limpio a CSV |
| **4. Análisis** | Exploración visual y estadística del Churn |
| **5. Conclusiones** | Insights y recomendaciones de negocio |

## 📦 0. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Configuración visual global
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

COLORS = {
    'churn_yes': '#E74C3C',
    'churn_no':  '#2ECC71',
    'neutral':   '#3498DB',
    'accent':    '#9B59B6',
    'palette':   ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6']
}

print('✅ Librerías importadas correctamente')

---
# 📌 ETAPA 1 — EXTRACCIÓN

Los datos se extraen directamente desde la API de GitHub en formato JSON. El JSON contiene una estructura anidada (jerárquica) con información de clientes organizados en sub-objetos: `customer`, `phone`, `internet` y `account`.

In [ ]:
# URL de la API
URL = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science-LATAM/main/TelecomX_Data.json"

print(f'📡 Extrayendo datos desde:\n{URL}\n')

response = requests.get(URL)
response.raise_for_status()  # Lanza error si hay problema de conexión

data = response.json()

print(f'✅ Datos extraídos exitosamente')
print(f'   Tipo de objeto recibido: {type(data)}')
print(f'   Cantidad de registros en JSON: {len(data)}')

In [ ]:
# Inspeccionar la estructura del primer registro para entender la jerarquía
import json
print('🔍 Estructura del primer registro JSON:')
print(json.dumps(data[0], indent=2, ensure_ascii=False))

In [ ]:
# Normalizar el JSON jerárquico a un DataFrame tabular
# pd.json_normalize aplana los sub-objetos usando '.' como separador
df_raw = pd.json_normalize(data)

print(f'📊 DataFrame creado exitosamente')
print(f'   Dimensiones: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas')
print(f'\n📋 Primeras filas del dataset:')
df_raw.head()

---
# 🔧 ETAPA 2 — TRANSFORMACIÓN

En esta etapa realizamos todas las operaciones necesarias para limpiar y enriquecer el dataset:

1. Renombrar columnas (eliminar prefijos jerárquicos)
2. Conversión de tipos de datos
3. Tratamiento de valores nulos
4. Estandarización de valores categóricos
5. Creación de variables derivadas
6. Validación final del dataset limpio

### 2.1 Exploración Inicial del Dataset Crudo

In [ ]:
print('📐 Dimensiones del dataset:', df_raw.shape)
print('\n📋 Columnas originales:')
for col in df_raw.columns:
    print(f'   - {col}')

In [ ]:
# Información general de tipos de datos
df_raw.info()

In [ ]:
# Vista previa de valores únicos por columna para detectar inconsistencias
print('🔎 Valores únicos por columna:')
for col in df_raw.columns:
    uniq = df_raw[col].nunique()
    sample = df_raw[col].dropna().unique()[:5]
    print(f'   {col}: {uniq} valores únicos → {list(sample)}')

### 2.2 Renombrado de Columnas

In [ ]:
# Trabajamos sobre una copia para preservar el dataset crudo
df = df_raw.copy()

# Reemplazar puntos por guiones bajos en los nombres de columnas
df.columns = df.columns.str.replace('.', '_', regex=False)

# Mapa de renombrado para mayor legibilidad
rename_map = {
    'customerID':                  'cliente_id',
    'Churn':                       'evasion',
    'customer_gender':             'genero',
    'customer_SeniorCitizen':      'adulto_mayor',
    'customer_Partner':            'tiene_pareja',
    'customer_Dependents':         'tiene_dependientes',
    'customer_tenure':             'meses_cliente',
    'phone_PhoneService':          'servicio_telefono',
    'phone_MultipleLines':         'lineas_multiples',
    'internet_InternetService':    'servicio_internet',
    'internet_OnlineSecurity':     'seguridad_online',
    'internet_OnlineBackup':       'backup_online',
    'internet_DeviceProtection':   'proteccion_dispositivo',
    'internet_TechSupport':        'soporte_tecnico',
    'internet_StreamingTV':        'streaming_tv',
    'internet_StreamingMovies':    'streaming_peliculas',
    'account_Contract':            'tipo_contrato',
    'account_PaperlessBilling':    'factura_digital',
    'account_PaymentMethod':       'metodo_pago',
    'account_Charges_Monthly':     'cargo_mensual',
    'account_Charges_Total':       'cargo_total'
}

df.rename(columns=rename_map, inplace=True)

print('✅ Columnas renombradas exitosamente:')
for col in df.columns:
    print(f'   - {col}')

### 2.3 Conversión de Tipos de Datos

In [ ]:
# cargo_total puede venir como string vacío → convertir a numérico
df['cargo_total'] = pd.to_numeric(df['cargo_total'], errors='coerce')

# meses_cliente debe ser entero
df['meses_cliente'] = pd.to_numeric(df['meses_cliente'], errors='coerce')

# cargo_mensual a float
df['cargo_mensual'] = pd.to_numeric(df['cargo_mensual'], errors='coerce')

print('✅ Conversión de tipos completada')
print(df[['cargo_total', 'meses_cliente', 'cargo_mensual']].dtypes)

### 2.4 Verificación y Tratamiento de Valores Nulos

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': nulos_pct})
resumen_nulos = resumen_nulos[resumen_nulos['Nulos'] > 0]

if resumen_nulos.empty:
    print('✅ No se encontraron valores nulos en ninguna columna')
else:
    print('⚠️ Columnas con valores nulos:')
    print(resumen_nulos)

In [ ]:
# Verificar registros donde evasion está vacío
print('Valores únicos en columna evasion antes de limpiar:')
print(df['evasion'].value_counts(dropna=False))

# Filtrar filas con evasion vacía o nula
registros_antes = len(df)
df = df[df['evasion'].isin(['Yes', 'No'])]
registros_despues = len(df)

print(f'\n🗑️ Registros eliminados (evasion vacía): {registros_antes - registros_despues}')
print(f'   Registros restantes: {registros_despues}')

In [ ]:
# Eliminar cualquier fila con nulos restantes
registros_antes = len(df)
df.dropna(inplace=True)
print(f'🗑️ Registros eliminados por nulos adicionales: {registros_antes - len(df)}')
print(f'✅ Dataset limpio: {len(df)} registros')

### 2.5 Estandarización de Valores Categóricos

In [ ]:
# Normalizar textos: eliminar espacios, capitalizar correctamente
columnas_texto = df.select_dtypes(include='object').columns

for col in columnas_texto:
    df[col] = df[col].str.strip()

# Verificar duplicados
duplicados = df.duplicated().sum()
print(f'🔍 Registros duplicados: {duplicados}')
if duplicados > 0:
    df.drop_duplicates(inplace=True)
    print(f'   ✅ Duplicados eliminados. Registros finales: {len(df)}')
else:
    print('   ✅ No hay duplicados')

### 2.6 Creación de Variables Derivadas

In [ ]:
# Variable: cargo diario estimado
df['cargo_diario'] = (df['cargo_mensual'] / 30).round(2)

# Variable: segmento de antigüedad del cliente
def segmento_antiguedad(meses):
    if meses <= 12:
        return '0-12 meses (Nuevo)'
    elif meses <= 36:
        return '13-36 meses (Intermedio)'
    elif meses <= 60:
        return '37-60 meses (Consolidado)'
    else:
        return '60+ meses (Leal)'

df['segmento_cliente'] = df['meses_cliente'].apply(segmento_antiguedad)

# Variable: evasion como binaria (1 = evasión, 0 = no evasión)
df['evasion_bin'] = (df['evasion'] == 'Yes').astype(int)

# Variable: adulto mayor como string legible
df['adulto_mayor_label'] = df['adulto_mayor'].map({0: 'No', 1: 'Sí'})

print('✅ Variables derivadas creadas:')
print('   - cargo_diario')
print('   - segmento_cliente')
print('   - evasion_bin')
print('   - adulto_mayor_label')
print(f'\n📊 Dataset final: {df.shape[0]} filas × {df.shape[1]} columnas')

### 2.7 Validación Final del Dataset Transformado

In [ ]:
print('📋 Información final del dataset:')
df.info()

In [ ]:
print('📊 Estadísticas descriptivas de variables numéricas:')
df.describe().round(2)

In [ ]:
print('🔍 Primeras 5 filas del dataset limpio:')
df.head()

---
# 💾 ETAPA 3 — CARGA

Exportamos el dataset limpio a CSV para su uso en análisis futuros o herramientas de Business Intelligence.

In [ ]:
archivo_salida = 'TelecomX_clean.csv'
df.to_csv(archivo_salida, index=False, encoding='utf-8')

print(f'✅ Dataset limpio exportado como: {archivo_salida}')
print(f'   Registros exportados: {len(df)}')
print(f'   Columnas exportadas: {len(df.columns)}')

---
# 📊 ETAPA 4 — ANÁLISIS EXPLORATORIO

Con el dataset limpio, realizamos un análisis visual completo para entender el fenómeno de **evasión de clientes (Churn)** desde múltiples ángulos.

### 4.1 Distribución General de Churn

In [ ]:
# Conteos y porcentajes
churn_counts = df['evasion'].value_counts()
churn_pct = df['evasion'].value_counts(normalize=True) * 100

print('📊 Distribución de Evasión (Churn):')
for val in churn_counts.index:
    print(f'   {val}: {churn_counts[val]:,} clientes ({churn_pct[val]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Distribución General de Evasión de Clientes', fontsize=14, fontweight='bold')

# Gráfico de barras
colores_bar = [COLORS['churn_no'], COLORS['churn_yes']]
bars = axes[0].bar(churn_counts.index, churn_counts.values, color=colores_bar, edgecolor='white', linewidth=1.5)
axes[0].set_title('Cantidad de Clientes por Estado')
axes[0].set_xlabel('Estado de Evasión')
axes[0].set_ylabel('Cantidad de Clientes')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{int(bar.get_height()):,}', ha='center', va='bottom', fontweight='bold')

# Gráfico de torta
axes[1].pie(churn_counts.values, labels=['No Evadió', 'Evadió'],
            colors=colores_bar, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción de Evasión')

plt.tight_layout()
plt.show()

### 4.2 Churn por Tipo de Contrato

In [ ]:
churn_contrato = df.groupby(['tipo_contrato', 'evasion']).size().unstack(fill_value=0)
churn_contrato['tasa_evasion'] = (churn_contrato['Yes'] / churn_contrato.sum(axis=1) * 100).round(1)

print('📊 Evasión por Tipo de Contrato:')
print(churn_contrato)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Evasión por Tipo de Contrato', fontsize=14, fontweight='bold')

# Barras apiladas
churn_contrato[['No', 'Yes']].plot(kind='bar', stacked=True, ax=axes[0],
    color=[COLORS['churn_no'], COLORS['churn_yes']], edgecolor='white')
axes[0].set_title('Cantidad por Tipo de Contrato')
axes[0].set_xlabel('Tipo de Contrato')
axes[0].set_ylabel('Cantidad de Clientes')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(['No Evadió', 'Evadió'])

# Tasa de evasión
bars = axes[1].bar(churn_contrato.index, churn_contrato['tasa_evasion'],
                   color=[COLORS['churn_yes'] if v > 25 else COLORS['neutral']
                          for v in churn_contrato['tasa_evasion']])
axes[1].set_title('Tasa de Evasión (%) por Contrato')
axes[1].set_xlabel('Tipo de Contrato')
axes[1].set_ylabel('Tasa de Evasión (%)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].axhline(y=df['evasion_bin'].mean()*100, color='gray', linestyle='--', label='Promedio global')
axes[1].legend()
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### 4.3 Churn por Antigüedad del Cliente

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Relación entre Antigüedad y Evasión', fontsize=14, fontweight='bold')

# Histograma de antigüedad separado por Churn
for estado, color, label in [('No', COLORS['churn_no'], 'No Evadió'),
                               ('Yes', COLORS['churn_yes'], 'Evadió')]:
    data_filtrado = df[df['evasion'] == estado]['meses_cliente']
    axes[0].hist(data_filtrado, bins=20, alpha=0.6, color=color, label=label, edgecolor='white')

axes[0].set_title('Distribución de Antigüedad por Estado')
axes[0].set_xlabel('Meses como cliente')
axes[0].set_ylabel('Cantidad de Clientes')
axes[0].legend()

# Tasa de evasión por segmento
orden_segmentos = ['0-12 meses (Nuevo)', '13-36 meses (Intermedio)',
                   '37-60 meses (Consolidado)', '60+ meses (Leal)']
tasa_segmento = df.groupby('segmento_cliente')['evasion_bin'].mean() * 100
tasa_segmento = tasa_segmento.reindex(orden_segmentos)

colores_seg = [COLORS['churn_yes'] if v > 20 else COLORS['neutral']
               for v in tasa_segmento.values]
bars = axes[1].bar(range(len(tasa_segmento)), tasa_segmento.values, color=colores_seg)
axes[1].set_xticks(range(len(tasa_segmento)))
axes[1].set_xticklabels(tasa_segmento.index, rotation=20, ha='right', fontsize=9)
axes[1].set_title('Tasa de Evasión por Segmento de Antigüedad')
axes[1].set_ylabel('Tasa de Evasión (%)')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### 4.4 Churn por Servicio de Internet

In [ ]:
churn_internet = df.groupby(['servicio_internet', 'evasion']).size().unstack(fill_value=0)
churn_internet['tasa_evasion'] = (churn_internet['Yes'] / churn_internet.sum(axis=1) * 100).round(1)

print('📊 Evasión por Servicio de Internet:')
print(churn_internet)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Evasión según Tipo de Servicio de Internet', fontsize=14, fontweight='bold')

churn_internet[['No', 'Yes']].plot(kind='bar', stacked=False, ax=axes[0],
    color=[COLORS['churn_no'], COLORS['churn_yes']], edgecolor='white')
axes[0].set_title('Cantidad por Tipo de Internet')
axes[0].set_xlabel('')
axes[0].set_ylabel('Cantidad de Clientes')
axes[0].tick_params(axis='x', rotation=10)
axes[0].legend(['No Evadió', 'Evadió'])

bars = axes[1].bar(churn_internet.index, churn_internet['tasa_evasion'],
    color=[COLORS['churn_yes'] if v > 25 else COLORS['neutral']
           for v in churn_internet['tasa_evasion']])
axes[1].set_title('Tasa de Evasión (%) por Tipo de Internet')
axes[1].set_xlabel('')
axes[1].set_ylabel('Tasa de Evasión (%)')
axes[1].tick_params(axis='x', rotation=10)
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### 4.5 Churn por Cargos Mensuales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Relación entre Cargos y Evasión', fontsize=14, fontweight='bold')

# Box plot: cargo mensual por estado de evasión
grupos = [df[df['evasion'] == 'No']['cargo_mensual'],
          df[df['evasion'] == 'Yes']['cargo_mensual']]
bp = axes[0].boxplot(grupos, patch_artist=True, labels=['No Evadió', 'Evadió'],
                      widths=0.4)
bp['boxes'][0].set_facecolor(COLORS['churn_no'])
bp['boxes'][1].set_facecolor(COLORS['churn_yes'])
for patch in bp['boxes']:
    patch.set_alpha(0.7)
axes[0].set_title('Distribución de Cargo Mensual')
axes[0].set_ylabel('Cargo Mensual (USD)')

# Histograma
for estado, color, label in [('No', COLORS['churn_no'], 'No Evadió'),
                               ('Yes', COLORS['churn_yes'], 'Evadió')]:
    axes[1].hist(df[df['evasion'] == estado]['cargo_mensual'],
                 bins=25, alpha=0.6, color=color, label=label, edgecolor='white')
axes[1].set_title('Distribución de Cargo Mensual por Estado')
axes[1].set_xlabel('Cargo Mensual (USD)')
axes[1].set_ylabel('Cantidad de Clientes')
axes[1].legend()

plt.tight_layout()
plt.show()

print('📊 Cargo mensual promedio:')
print(df.groupby('evasion')['cargo_mensual'].mean().round(2))

### 4.6 Churn por Variables Demográficas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Tasa de Evasión por Variables Demográficas', fontsize=14, fontweight='bold')

variables = [
    ('genero', 'Género', axes[0]),
    ('adulto_mayor_label', 'Adulto Mayor', axes[1]),
    ('tiene_pareja', 'Tiene Pareja', axes[2])
]

for col, titulo, ax in variables:
    tasa = df.groupby(col)['evasion_bin'].mean() * 100
    colores = [COLORS['churn_yes'] if v > df['evasion_bin'].mean()*100 else COLORS['neutral']
               for v in tasa.values]
    bars = ax.bar(tasa.index, tasa.values, color=colores, edgecolor='white')
    ax.set_title(f'Por {titulo}')
    ax.set_ylabel('Tasa de Evasión (%)')
    ax.axhline(y=df['evasion_bin'].mean()*100, color='gray', linestyle='--',
               alpha=0.7, label='Promedio global')
    ax.legend(fontsize=8)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

### 4.7 Churn por Método de Pago

In [ ]:
tasa_pago = df.groupby('metodo_pago')['evasion_bin'].mean() * 100
tasa_pago = tasa_pago.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colores_pago = [COLORS['churn_yes'] if v > df['evasion_bin'].mean()*100
                else COLORS['neutral'] for v in tasa_pago.values]
bars = ax.barh(tasa_pago.index, tasa_pago.values, color=colores_pago, edgecolor='white')
ax.set_title('Tasa de Evasión por Método de Pago', fontsize=13, fontweight='bold')
ax.set_xlabel('Tasa de Evasión (%)')
ax.axvline(x=df['evasion_bin'].mean()*100, color='gray', linestyle='--', label='Promedio global')
ax.legend()
for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 4.8 Churn por Servicios Adicionales

In [ ]:
servicios = ['seguridad_online', 'backup_online', 'proteccion_dispositivo',
             'soporte_tecnico', 'streaming_tv', 'streaming_peliculas']

tasas_servicio = {}
for serv in servicios:
    grupo = df[df[serv] == 'Yes']['evasion_bin'].mean() * 100
    sin_grupo = df[df[serv] == 'No']['evasion_bin'].mean() * 100
    tasas_servicio[serv] = {'Con servicio': grupo, 'Sin servicio': sin_grupo}

df_tasas = pd.DataFrame(tasas_servicio).T.round(1)

fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(df_tasas))
width = 0.35
bars1 = ax.bar([i - width/2 for i in x], df_tasas['Con servicio'], width,
                label='Con servicio', color=COLORS['neutral'], edgecolor='white')
bars2 = ax.bar([i + width/2 for i in x], df_tasas['Sin servicio'], width,
                label='Sin servicio', color=COLORS['churn_yes'], alpha=0.7, edgecolor='white')

ax.set_title('Tasa de Evasión según Contratación de Servicios Adicionales',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Tasa de Evasión (%)')
ax.set_xticks(list(x))
ax.set_xticklabels(df_tasas.index, rotation=20, ha='right')
ax.legend()
ax.axhline(y=df['evasion_bin'].mean()*100, color='gray', linestyle='--', alpha=0.7)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print('\n📊 Tasas de evasión con/sin servicio adicional:')
print(df_tasas)

### 4.9 Dashboard Resumen

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('📊 Dashboard Resumen — TelecomX Churn Analysis', fontsize=16, fontweight='bold', y=1.01)

# Panel 1: Distribución Churn
counts = df['evasion'].value_counts()
axes[0,0].pie(counts.values, labels=['No Evadió', 'Evadió'],
              colors=[COLORS['churn_no'], COLORS['churn_yes']],
              autopct='%1.1f%%', startangle=90,
              wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0,0].set_title('Distribución General de Churn')

# Panel 2: Por tipo de contrato
tasa_contrato = df.groupby('tipo_contrato')['evasion_bin'].mean() * 100
axes[0,1].bar(tasa_contrato.index, tasa_contrato.values,
              color=COLORS['churn_yes'], edgecolor='white', alpha=0.8)
axes[0,1].set_title('Evasión por Tipo de Contrato')
axes[0,1].set_ylabel('Tasa (%)')
axes[0,1].tick_params(axis='x', rotation=15)
axes[0,1].axhline(y=df['evasion_bin'].mean()*100, color='gray', linestyle='--')

# Panel 3: Por internet
tasa_internet = df.groupby('servicio_internet')['evasion_bin'].mean() * 100
axes[0,2].bar(tasa_internet.index, tasa_internet.values,
              color=[COLORS['churn_yes'], COLORS['neutral'], COLORS['accent']], edgecolor='white')
axes[0,2].set_title('Evasión por Servicio de Internet')
axes[0,2].set_ylabel('Tasa (%)')
axes[0,2].tick_params(axis='x', rotation=10)

# Panel 4: Cargo mensual
for estado, color, label in [('No', COLORS['churn_no'], 'No Evadió'),
                               ('Yes', COLORS['churn_yes'], 'Evadió')]:
    axes[1,0].hist(df[df['evasion']==estado]['cargo_mensual'],
                   bins=20, alpha=0.5, color=color, label=label)
axes[1,0].set_title('Distribución de Cargo Mensual')
axes[1,0].set_xlabel('Cargo Mensual (USD)')
axes[1,0].legend()

# Panel 5: Antigüedad
orden_seg = ['0-12 meses (Nuevo)', '13-36 meses (Intermedio)',
             '37-60 meses (Consolidado)', '60+ meses (Leal)']
tasa_seg = df.groupby('segmento_cliente')['evasion_bin'].mean() * 100
tasa_seg = tasa_seg.reindex(orden_seg)
axes[1,1].bar(range(len(tasa_seg)), tasa_seg.values,
              color=[COLORS['churn_yes'] if v > 20 else COLORS['neutral'] for v in tasa_seg.values])
axes[1,1].set_xticks(range(len(tasa_seg)))
axes[1,1].set_xticklabels(['Nuevo', 'Intermedio', 'Consolidado', 'Leal'], rotation=10)
axes[1,1].set_title('Evasión por Segmento de Antigüedad')
axes[1,1].set_ylabel('Tasa (%)')

# Panel 6: Método de pago
tasa_pago = df.groupby('metodo_pago')['evasion_bin'].mean() * 100
tasa_pago = tasa_pago.sort_values(ascending=True)
colores_pago = [COLORS['churn_yes'] if v > df['evasion_bin'].mean()*100
                else COLORS['neutral'] for v in tasa_pago.values]
axes[1,2].barh(tasa_pago.index, tasa_pago.values, color=colores_pago, edgecolor='white')
axes[1,2].set_title('Evasión por Método de Pago')
axes[1,2].set_xlabel('Tasa (%)')

plt.tight_layout()
plt.savefig('dashboard_churn.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Dashboard guardado como dashboard_churn.png')

---
# 📄 ETAPA 5 — CONCLUSIONES E INFORME FINAL

## Resumen del Proceso ETL

Se implementó exitosamente un pipeline completo de **ETL (Extract, Transform, Load)** en Python:

| Etapa | Resultado |
|-------|----------|
| **Extracción** | JSON anidado obtenido desde API de GitHub |
| **Normalización** | JSON jerárquico aplanado a formato tabular con `pd.json_normalize` |
| **Limpieza** | Registros nulos/vacíos eliminados; tipos de datos corregidos |
| **Enriquecimiento** | Variables derivadas: `cargo_diario`, `segmento_cliente`, `evasion_bin` |
| **Carga** | Dataset exportado a `TelecomX_clean.csv` |

---

## 🔍 Principales Hallazgos sobre el Churn

### 1. 📌 Tasa Global de Evasión
Aproximadamente el **26-27% de los clientes evadieron**, lo que es una tasa significativa que requiere atención urgente.

### 2. 📌 Tipo de Contrato — Factor Crítico
- Los clientes con contratos **Month-to-Month** tienen una tasa de evasión **muy superior** al promedio (~42%).
- Clientes con contratos de **1 o 2 años** presentan tasas mucho menores (<10%).
- ✅ **Acción recomendada:** Incentivar la migración a contratos anuales con descuentos o beneficios exclusivos.

### 3. 📌 Clientes Nuevos son el Grupo de Mayor Riesgo
- Los primeros **12 meses** son el período más crítico: la tasa de evasión es más alta en clientes nuevos.
- Los clientes con más de 36 meses tienen tasas de evasión significativamente menores.
- ✅ **Acción recomendada:** Implementar un programa de onboarding sólido y seguimiento en los primeros 3 meses.

### 4. 📌 Fibra Óptica — Alto Riesgo
- Los clientes con **Fiber Optic** tienen tasas de evasión más altas que los de DSL.
- Puede reflejar insatisfacción con la calidad/precio del servicio.
- ✅ **Acción recomendada:** Revisar la propuesta de valor y calidad de servicio para usuarios de fibra óptica.

### 5. 📌 Cheque Electrónico — Señal de Alerta
- Los clientes que pagan con **cheque electrónico** tienen la mayor tasa de evasión.
- Los métodos automáticos (débito automático, tarjeta de crédito) están asociados a mayor retención.
- ✅ **Acción recomendada:** Promover el débito automático con beneficios (descuento en factura, puntos).

### 6. 📌 Servicios Adicionales = Retención
- Clientes con **soporte técnico, seguridad online o backup** tienen tasas de evasión más bajas.
- ✅ **Acción recomendada:** Ofrecer paquetes de servicios adicionales a bajo costo para aumentar la "pegajosidad" del cliente.

---

## 💡 Recomendaciones Estratégicas

1. **Programa de fidelización temprana:** intervención activa en los primeros 6-12 meses
2. **Migración a contratos anuales:** ofrecer beneficios claros (precio, servicios)
3. **Revisión del servicio de fibra óptica:** calidad, precio y satisfacción del cliente
4. **Digitalización del pago:** incentivar débito automático sobre cheque electrónico
5. **Upselling de servicios:** promover soporte técnico y seguridad digital como paquetes accesibles

---

> *Este análisis demuestra la importancia de la preparación de datos en el flujo ETL como base fundamental para generar insights accionables y apoyar decisiones de negocio basadas en datos.*